In [2]:
import pandas as pd

nome_file = 'children-out-school.csv' 
df = pd.read_csv('../../datasets/raw/children-out-school/children-out-school.csv')

colonne_da_tenere = [
    "iso_code", 
    "region_group", 
    "country", 
    "year", 
    "category", 
    "region", 
    "comp_prim_v2_m", 
    "eduout_prim_m"
]

df_filtrato = df[colonne_da_tenere]
print(df_filtrato.head())

  iso_code               region_group      country  year   category region  \
0      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
1      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
2      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
3      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
4      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   

   comp_prim_v2_m  eduout_prim_m  
0          0.3489            NaN  
1          0.6573            NaN  
2          0.4849            NaN  
3          0.3656            NaN  
4          0.4102            NaN  


/var/folders/nj/kxr7k46n76q3w6dmc0rj_53r0000gn/T/ipykernel_35079/3447323748.py:4: DtypeWarning: Columns (7,12,13,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../datasets/raw/children-out-school/children-out-school.csv')


In [3]:
regioni_target = ['Northern Africa and Western Asia', 'Sub-Saharan Africa']
categoria_target = 'Region'

df_filtrato_africa_asia = df_filtrato[
    (df_filtrato['region_group'].isin(regioni_target)) & 
    (df_filtrato['category'] == categoria_target)
]
print(df_filtrato_africa_asia.head())

#df_filtrato_africa_asia.to_csv('dataset_africa_asia.csv', index=False)

    iso_code        region_group country  year category          region  \
913      AGO  Sub-Saharan Africa  Angola  2015   Region           Bengo   
914      AGO  Sub-Saharan Africa  Angola  2015   Region        Benguela   
915      AGO  Sub-Saharan Africa  Angola  2015   Region             Bie   
916      AGO  Sub-Saharan Africa  Angola  2015   Region         Cabinda   
917      AGO  Sub-Saharan Africa  Angola  2015   Region  Cuando Cubango   

     comp_prim_v2_m  eduout_prim_m  
913          0.5377         0.3201  
914          0.6168         0.2069  
915          0.4104         0.3247  
916          0.6905         0.1712  
917          0.2633         0.4733  


In [5]:
df_continenti = pd.read_csv('../../datasets/raw/country-codes/country-code.csv')
df_continenti.drop_duplicates(subset=['Three_Letter_Country_Code'], inplace=True)

# left_on e right_on si usano quando le colonne chiave hanno nomi diversi
df_unito = pd.merge(
    df_filtrato_africa_asia, 
    df_continenti[['Three_Letter_Country_Code', 'Continent_Name']], 
    left_on='iso_code', 
    right_on='Three_Letter_Country_Code', 
    how='left'
)

df_africa = df_unito[df_unito['Continent_Name'] == 'Africa'].copy()

df_africa.drop(columns=['region_group','category','Continent_Name','Three_Letter_Country_Code'], inplace=True)
df_africa.to_csv('out-of-school.csv', index=False)

In [ ]:
import pandas as pd

df = pd.read_csv('dataset-school-clean.csv', sep=';')

df.drop(columns=['out_primary'], inplace=True)
# Converti le colonne numeriche (gestisce eventuali valori non numerici)
df['compl_primary'] = pd.to_numeric(df['compl_primary'], errors='coerce')

# Raggruppa per iso_code, country, region e calcola la media
df_grouped = (
    df.groupby(['iso_code', 'country', 'region'], as_index=False)
    .agg(
        compl_primary_avg=('compl_primary', 'mean'),
    )
    .round(4)
)

print(f"Righe originali:   {len(df)}")
print(f"Righe raggruppate: {len(df_grouped)}")
print(df_grouped.head(20))